# 1. Set Paths

In [1]:
import os

In [17]:
# LORAKS test

DATASET_PATH = "/data/p_03002/data"

SOURCE_PATH = os.path.join(DATASET_PATH, "source")
PREPARED_PATH = os.path.join(DATASET_PATH, "temp/LORAKS")
BIDSIFIED_PATH = os.path.join(DATASET_PATH, "bids/derivatives/LORAKS")
RESOURCES_PATH = os.path.join(DATASET_PATH, "bids/code/resources")
WORKING_DIR = os.getcwd()

In [18]:
print("Dataset path:", os.path.isdir(DATASET_PATH))
print("Source path:", os.path.isdir(SOURCE_PATH))
print("Prepared path:", os.path.isdir(PREPARED_PATH))
print("Bidsified path:", os.path.isdir(BIDSIFIED_PATH))
print("Resources path:", os.path.isdir(RESOURCES_PATH))
print("Working directory:", WORKING_DIR)

Dataset path: True
Source path: True
Prepared path: True
Bidsified path: True
Resources path: True
Working directory: /data/u_kuegler_software/git/MPM_bidsification


# 2. Initialize `bidsme` and get the `logger` object
which which will control the logging of all bidsme functions:will control the logging of all bidsme functions:

In [19]:
import bidsme
logger = bidsme.init()

main(81) - INFO 
main(82) - INFO -------------- START bidsme ----------------
main(83) - INFO Tue Jan  7 15:19:43 2025
main(84) - INFO version: 1.8.1
bidsme.schema.BIDSschema(670) - INFO Loaded BIDS schema version 1.10.0


# 3. Prepare data set for bidsification

In [20]:
# help(bidsme.prepare)

In [30]:
logger.setLevel("INFO")
# bidsme.prepare(SOURCE_PATH, PREPARED_PATH, 
#                data_dirs={"nii/localizer*":"MRI",
#                           "nii/calc_shims_40mm*":"MRI",
#                           "nii/ernst_kp_mtflash3d_*_0p5_sag_*":"MRI",
#                           "nii/kp_afib1*":"MRI",
#                           "nii/AAHead*":"MRI",
#                           "nii/cmrr_noddi*":"MRI",
#                           "nii/t1w_kp_mtflash3d_*_0p6_*":"MRI",
#                           "nii/pdw_kp_mtflash3d_*_0p6_*":"MRI",
#                           "nii/mtw_kp_mtflash3d_*_0p6_*":"MRI",
#                           "nii/semc_js_res0p6_*":"MRI",
#                           "nii/tfl_multiMTC*":"MRI",
#                           "nii/ke_gre_clearswi*":"MRI",},
#                plugin_file="plugins_bidsme/plugin_prepare_nk.py",
#               # sub_list=["sub-41006.a1"]) # only run on specified subjects (must be specified in BIDS notation)
#               )         
bidsme.prepare(SOURCE_PATH, PREPARED_PATH, 
               data_dirs={"nii_loraks":"MRI",
                          # "nii_loraks/ernst_loraks":"MRI",
                          # "nii_loraks/pdw_loraks":"MRI",
                          # "nii_loraks/t1w_loraks":"MRI",
                          },
               plugin_file=os.path.join(RESOURCES_PATH, "plugins/plugin_prepare_loraks_nk.py"),
               part_template=os.path.join(RESOURCES_PATH,"table_templates/participants_nk.json"),
               # sub_list=['sub-008']
              )                           
bidsme.tools.info.reporterrors(logger)
bidsme.tools.info.reseterrors(logger)

bidsme.prepare(192) - INFO -------------- Prepearing data -------------
bidsme.prepare(193) - INFO Source directory: /data/pt_03002/data/source
bidsme.prepare(194) - INFO Destination directory: /data/pt_03002/data/temp/LORAKS
bidsme.plugins.plugins(79) - INFO Loading module plugin_prepare_loraks_nk from /data/pt_03002/data/bids/code/resources/plugins/plugin_prepare_loraks_nk.py
Loading sessions_nk.json from /data/pt_03002/data/bids/code/resources/table_templates/sessions_nk.json.
          This functionality is not part of Bidsme, but implemented in a plugin. 
          It only works for processing all sessions of a subjects. Problems may 
          arise if the plugin is used for single sessions.
bidsme.bidsMeta.BidsTable(141) - INFO Created empty participants.tsv table
Subject ID derived from '/data/pt_03002/data/id_info/subject_ids.csv'.
Current subject: 15484.08 -> 005
bidsme.prepare(295) - INFO Scanning folder /data/pt_03002/data/source/15484.08/20241107
Session ID derived from '/

> **Note**: **apparently, it is not possible to specify a specific subset of sessions**
> + If the user wishes to rename subjects and/or sessions, it can be done with plug-in functions ```SubjectEP``` and ```SessionEP``` or by renaming directly folders in the prepared dataset.

# 4. Create the bidsmap.yaml

+ most tedious part of the process

In [ ]:
# help(bidsme.mapper)

In [33]:
PLUGIN_BIDS=os.path.join(RESOURCES_PATH, "plugins/plugin_bidsify_loraks_nk.py")

In [ ]:
bidsme.mapper(PREPARED_PATH, BIDSIFIED_PATH, plugin_file=PLUGIN_BIDS)
bidsme.tools.info.reporterrors(logger)
bidsme.tools.info.reseterrors(logger)

+ open the created yaml file in VS Code
+ fix each warning/error, save the file, and repeat the code block above
    - find help at in the [Jupyter Notebooks in the bidsme tutorial](https://github.com/CyclotronResearchCentre/bidsme_tutorial) or in the docs [under Bidsmap creation](https://github.com/CyclotronResearchCentre/bidsme/blob/dev/doc/creating_map.md)
+ resume until there are no warnings left

> **Note:** to find a specific string in a file, use the command ```cat file.json | grep -i "string"``` **or** use ```less file.json``` and search using ```/string``` (```n``` will take you to the next entry & ```shift+n``` to the previous one; ```-I``` for case-insensitive)

> **Note:** Bidsme allow some limited transformation of data retrieved from header, these transformations are called actions and are defined in function ```action_value``` in file ```INSTALLATION_PATH/bidsme/Modules/common.py```.

```
Accepted actions:
    "": no action, return value
    int: cast value to int
    float: cast value to float
    str: cast value to string
    format<parameters>: apply python3 formatting
        mini-language to value, {:<parameters>}.format(value)
    scale<int>: apply a 10-based scale to value,
        value ** <int>
    mult<float>: multiply value
    div<float>: divide value
    round<int>: round value to given precision
```


+ The naming schema and sidecar json fields for a given modality (in this case MRI) are defined in $INSTALLATION_PATH/bidsme/Modules/MRI/_MRI.py. The list of entities is stored in modalities dictionary. If an image belongs for example to anat, bidsme will load the list of entities from modalities["anat"].

+ The optional model field will foce to use different list of entities from modalities dictionary. We will use the models extensively, while creating map for MPM part of the examle dataset.


# 5. Bidsification of the data set

In [46]:
MAP_FILE = os.path.join(BIDSIFIED_PATH, "code/bidsme/bidsmap.yaml")
PLUGIN_BIDS=os.path.join(RESOURCES_PATH, "plugins/plugin_bidsify_loraks_nk.py")

-b = mapping file, --plugin = plugin

In [ ]:
!bidsme bidsify $PREPARED_PATH $BIDSIFIED_PATH -b $MAP_FILE --plugin $PLUGIN_BIDS
# !bidsme bidsify $PREPARED_PATH $BIDSIFIED_PATH -b $MAP_FILE --plugin $PLUGIN_FILE_BIDS --participants 'sub-004'
# !bidsme bidsify $PREPARED_PATH $BIDSIFIED_PATH -b $MAP_FILE --plugin $PLUGIN_FILE_BIDS --skip-existing

## Hints


The `004-al_mtflash3d_PDw` and `005-al_mtflash3d_PDw` are
the anatomical images
(suffix -- `MPM`)
taken using several echo times (`echo-1` ... `echo-6`),
and splitted into magnitude and phase components (`part-mag` and `part-phase`).
Additionaly, as for the PD-weighted images, the MT pulse wasn't used, we will
add the `mt-off` entity.
We will also add `flip-1` to the name, to mark that PDw images uses different
flip angle from T
So the final name will become:
`anat/sub-001_ses-s01530_acq-PDw_echo-1_mt-off_part-mag_MPM.nii`.
`anat/sub-001_ses-s01530_acq-PDw_echo-1_flip-1_mt-off_part-mag_MPM.nii`

The `002-al_mtflash3d_sensArray` and `003-al_mtflash3d_sensBody`
are [B1 fieldmaps](https://bids-specification.readthedocs.io/en/stable/99-appendices/11-qmri.html#rb1cor-specific-notes)
(suffix -- `RB1COR`),
taken for the PD-weighted images, using head and body coils
(`acq-headPDw` and `acq-bodyPDw`).
So their names will be simply: `fmap/sub-001_ses-s01530_acq-headPDw_RB1COR.nii`.

The similar names can be applied to T1w and MTw images and corresponding fieldmaps,
using corresponding `acq-` entities `acq-T1w` and `acq-MTw`.
For MTw images we alse need to use the `mt-on` entity.

Finally, `014-al_B1mapping` is the
[global B1 map](https://bids-specification.readthedocs.io/en/stable/99-appendices/11-qmri.html#tb1epi-specific-notes)
(suffix -- `TB1EPI`)
sets of images, taken with two echo times (`echo-1`, `echo-2`)
and several flip angles (`flip-01`, ... `flip-08`).
So the final name will become: `fmap/sub-001_ses-s01530_echo-1_flip-01_TB1EPI`

# Testing

In [56]:
import pandas as pd
import os
import numpy as np

In [62]:
csv_file = "subject_ids.csv"

if not os.path.isfile(csv_file):
    id_dict = {'subject_IDs': [], 
               'BIDS_IDs': []
               }
    id_df = pd.DataFrame(id_dict)
else:
    id_df = pd.read_csv(csv_file).astype(str)


subject = '123457.53'

if subject in id_df['subject_IDs'].values:
    subject = f"{int(id_df.loc[id_df['subject_IDs'] == subject, 'BIDS_IDs'].iloc[0]):03}"
    print(f'subject was changed to {subject}.')
else:
    print(f"Subject ID not present in '{csv_file}'. Adding and indexing the subject.")
    if id_df.empty:
        current_bidsID = 1
    else:
        # current_bidsID = int(id_df['BIDS_IDs'].iat[-1]) + 1
        current_bidsID = int(np.max(id_df['BIDS_IDs'].astype(int))) +1
    print(f"current ID: {current_bidsID}")

    new_row = pd.DataFrame({'subject_IDs': [subject], 'BIDS_IDs': [str(current_bidsID)]})
    display(new_row)
    id_df = pd.concat([id_df, new_row], ignore_index=True)

    subject = f"{int(new_row['BIDS_IDs'].iloc[0]):03}"
    print(subject)

display(id_df)

# Store the DataFrame in a CSV file
id_df.to_csv(csv_file, index=False)

subject was changed to 001.


,subject_IDs,BIDS_IDs
0,123457.53,1


In [ ]:
a = ['5', '3', '1']

inta = int(np.max(a.astype(int)))
print(type(inta))

aa = np.max(a.astype(int))
print(type(aa))

In [91]:
import os

source = "/data/pt_02262/bids_test/bids_folders_autom/source"
print(os.path.dirname(source))

/data/pt_02262/bids_test/bids_folders_autom
